# Fine-tune DistilBERT for Seven-Emotion YouTube Comment Analysis

This notebook fine-tunes `distilbert-base-uncased` for seven emotion classes: anger, disgust, fear, joy, neutral, sadness, and surprise.


## 1. Install dependencies


In [ ]:
!pip install -q transformers datasets accelerate evaluate scikit-learn pandas pyarrow


## 2. Import packages and define labels


In [ ]:
import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments

TARGET_LABELS = ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]
LABEL_TO_ID = {label: idx for idx, label in enumerate(TARGET_LABELS)}
ID_TO_LABEL = {idx: label for label, idx in LABEL_TO_ID.items()}

TARGET_LABELS


## 3. Load and prepare GoEmotions dataset

The raw dataset is multi-label with 28 emotion columns. We keep clean single-label examples where exactly one label is active and the active label is one of our seven target emotions.


In [ ]:
PARQUET_URLS = {
    "train": "https://huggingface.co/datasets/SetFit/go_emotions/resolve/refs%2Fconvert%2Fparquet/default/train/0000.parquet",
    "validation": "https://huggingface.co/datasets/SetFit/go_emotions/resolve/refs%2Fconvert%2Fparquet/default/validation/0000.parquet",
    "test": "https://huggingface.co/datasets/SetFit/go_emotions/resolve/refs%2Fconvert%2Fparquet/default/test/0000.parquet",
}

def filter_single_target_emotions(df):
    all_label_columns = [column for column in df.columns if column != "text"]
    single_label_df = df[df[all_label_columns].sum(axis=1) == 1].copy()
    target_df = single_label_df[single_label_df[TARGET_LABELS].sum(axis=1) == 1].copy()
    target_df["label_name"] = target_df[TARGET_LABELS].idxmax(axis=1)
    target_df["label"] = target_df["label_name"].map(LABEL_TO_ID).astype(int)
    return target_df[["text", "label_name", "label"]].reset_index(drop=True)

def balance_by_label(df, random_state=42):
    min_count = df["label_name"].value_counts().min()
    parts = []
    for label_name in sorted(df["label_name"].unique()):
        parts.append(df[df["label_name"] == label_name].sample(n=min_count, random_state=random_state))
    return pd.concat(parts, ignore_index=True).sample(frac=1.0, random_state=random_state).reset_index(drop=True)

prepared_frames = {}
for split, url in PARQUET_URLS.items():
    raw_df = pd.read_parquet(url)
    filtered_df = filter_single_target_emotions(raw_df)
    prepared_frames[split] = balance_by_label(filtered_df)
    print(split, prepared_frames[split].shape)
    print(prepared_frames[split]["label_name"].value_counts().sort_index())


## 4. Convert pandas DataFrames to Hugging Face Datasets


In [ ]:
dataset = DatasetDict({
    split: Dataset.from_pandas(df[["text", "label"]], preserve_index=False)
    for split, df in prepared_frames.items()
})

dataset


## 5. Tokenize text


In [ ]:
BASE_MODEL = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

tokenized_dataset = dataset.map(tokenize_batch, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(["text"])
tokenized_dataset.set_format("torch")

tokenized_dataset


## 6. Load model


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(TARGET_LABELS),
    id2label=ID_TO_LABEL,
    label2id=LABEL_TO_ID,
)


## 7. Define metrics


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted",
        zero_division=0,
    )
    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


## 8. Train


In [ ]:
training_args = TrainingArguments(
    output_dir="./youtube-emotion-distilbert-results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()


## 9. Evaluate on validation and test sets


In [ ]:
validation_metrics = trainer.evaluate(tokenized_dataset["validation"])
test_metrics = trainer.evaluate(tokenized_dataset["test"])

print("Validation metrics:", validation_metrics)
print("Test metrics:", test_metrics)


## 10. Save model locally


In [ ]:
MODEL_OUTPUT_DIR = "./youtube-emotion-distilbert"
trainer.save_model(MODEL_OUTPUT_DIR)
tokenizer.save_pretrained(MODEL_OUTPUT_DIR)
print(f"Saved model to {MODEL_OUTPUT_DIR}")


## 11. Test the fine-tuned model with pipeline


In [ ]:
from transformers import pipeline

emotion_pipeline = pipeline("text-classification", model=MODEL_OUTPUT_DIR, tokenizer=MODEL_OUTPUT_DIR)
samples = [
    "I love this video so much!",
    "This campaign makes me angry.",
    "I am shocked by this announcement.",
    "This is just a normal update.",
]
emotion_pipeline(samples, truncation=True, return_token_type_ids=False)


## 12. YouTube domain adaptation

This optional second training stage continues from the GoEmotions fine-tuned model and adapts it to YouTube-style comments. The current project dataset contains 2,962 assistant-assisted YouTube comments from 30 videos, split into 2,370 training rows and 592 validation rows. The labels are used only for domain adaptation. The independent app benchmark in the report uses a separate manually reviewed 150-comment set.


In [ ]:
YOUTUBE_DATA_BASE = "https://raw.githubusercontent.com/chasezhang1999/youtube-emotion-analyzer/main/data/youtube_domain_7class_assistant"

youtube_train_df = pd.read_csv(f"{YOUTUBE_DATA_BASE}/train.csv")
youtube_val_df = pd.read_csv(f"{YOUTUBE_DATA_BASE}/validation.csv")

for df in [youtube_train_df, youtube_val_df]:
    df["text"] = df["text"].astype(str)
    df["label_name"] = df["label"].astype(str)
    df["label"] = df["label_id"].astype(int)

print("YouTube-domain train:", youtube_train_df.shape)
print(youtube_train_df["label_name"].value_counts().sort_index())
print("YouTube-domain validation:", youtube_val_df.shape)
print(youtube_val_df["label_name"].value_counts().sort_index())


In [ ]:
domain_dataset = DatasetDict({
    "train": Dataset.from_pandas(youtube_train_df[["text", "label"]], preserve_index=False),
    "validation": Dataset.from_pandas(youtube_val_df[["text", "label"]], preserve_index=False),
})

domain_tokenized_dataset = domain_dataset.map(tokenize_batch, batched=True)
domain_tokenized_dataset = domain_tokenized_dataset.remove_columns(["text"])
domain_tokenized_dataset.set_format("torch")

domain_tokenized_dataset


In [ ]:
domain_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_OUTPUT_DIR,
    num_labels=len(TARGET_LABELS),
    id2label=ID_TO_LABEL,
    label2id=LABEL_TO_ID,
)

domain_training_args = TrainingArguments(
    output_dir="./youtube-emotion-distilbert-domain-results",
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=25,
    report_to="none",
)

domain_trainer = Trainer(
    model=domain_model,
    args=domain_training_args,
    train_dataset=domain_tokenized_dataset["train"],
    eval_dataset=domain_tokenized_dataset["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

domain_trainer.train()


In [ ]:
domain_validation_metrics = domain_trainer.evaluate(domain_tokenized_dataset["validation"])
print("YouTube-domain validation metrics:", domain_validation_metrics)


In [ ]:
DOMAIN_MODEL_OUTPUT_DIR = "./youtube-emotion-distilbert-domain-adapted"
domain_trainer.save_model(DOMAIN_MODEL_OUTPUT_DIR)
tokenizer.save_pretrained(DOMAIN_MODEL_OUTPUT_DIR)
print(f"Saved domain-adapted model to {DOMAIN_MODEL_OUTPUT_DIR}")


In [ ]:
domain_pipeline = pipeline("text-classification", model=DOMAIN_MODEL_OUTPUT_DIR, tokenizer=DOMAIN_MODEL_OUTPUT_DIR)
domain_pipeline(samples, truncation=True, return_token_type_ids=False)


## 13. Upload to Hugging Face

Before running this section, create a Hugging Face write token and add it to Colab Secrets as `HF_TOKEN`.


In [ ]:
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)

repo_name = "chase1zhang/youtube-emotion-distilbert"
trainer.model.push_to_hub(repo_name)
tokenizer.push_to_hub(repo_name)
print(f"Uploaded original fine-tuned model to https://huggingface.co/{repo_name}")

domain_repo_name = "chase1zhang/youtube-emotion-distilbert-domain-adapted"
domain_trainer.model.push_to_hub(domain_repo_name)
tokenizer.push_to_hub(domain_repo_name)
print(f"Uploaded domain-adapted model to https://huggingface.co/{domain_repo_name}")


## 14. Verify uploaded Hugging Face models

Run this cell after uploading. It reloads the model from Hugging Face, not from the local Colab folder.


In [ ]:
uploaded_pipeline = pipeline("text-classification", model=repo_name, tokenizer=repo_name)
print("Original fine-tuned model:")
print(uploaded_pipeline(samples, truncation=True, return_token_type_ids=False))

uploaded_domain_pipeline = pipeline("text-classification", model=domain_repo_name, tokenizer=domain_repo_name)
print("Domain-adapted model:")
print(uploaded_domain_pipeline(samples, truncation=True, return_token_type_ids=False))
